In [8]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

PROJECT_DIR   = Path().resolve().parent
RAW_DIR       = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
DB_PATH       = PROJECT_DIR / "data" / "db" / "bluestock_mf.db"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: D:\Engsci_Study_materials\INTERNSHIP\Capstone_Project\data\db\bluestock_mf.db


In [9]:
DATASETS = {
    "fund_master"          : "01_fund_master.csv",
    "nav_history"          : "02_nav_history.csv",
    "aum_by_fund_house"    : "03_aum_by_fund_house.csv",
    "monthly_sip_inflows"  : "04_monthly_sip_inflows.csv",
    "category_inflows"     : "05_category_inflows.csv",
    "industry_folio_count" : "06_industry_folio_count.csv",
    "scheme_performance"   : "07_scheme_performance.csv",
    "investor_transactions": "08_investor_transactions.csv",
    "portfolio_holdings"   : "09_portfolio_holdings.csv",
    "benchmark_indices"    : "10_benchmark_indices.csv",
}

DATE_COLS = {
    "fund_master": ["launch_date"], "nav_history": ["date"],
    "aum_by_fund_house": ["date"], "monthly_sip_inflows": ["month"],
    "category_inflows": ["month"], "industry_folio_count": ["month"],
    "investor_transactions": ["transaction_date"],
    "portfolio_holdings": ["portfolio_date"], "benchmark_indices": ["date"],
}

raw = {}
for name, fname in DATASETS.items():
    path = RAW_DIR / fname
    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        for col in DATE_COLS.get(name, []):
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")
        raw[name] = df
        print(f"  {name:<30} {df.shape[0]:>7,} rows x {df.shape[1]} cols")
    else:
        print(f"  {name:<30} FILE NOT FOUND")
print(f"Loaded {len(raw)}/{len(DATASETS)} datasets")

  fund_master                         40 rows x 15 cols
  nav_history                     46,000 rows x 3 cols
  aum_by_fund_house                   90 rows x 5 cols
  monthly_sip_inflows                 48 rows x 6 cols
  category_inflows                   144 rows x 3 cols
  industry_folio_count                21 rows x 6 cols
  scheme_performance                  40 rows x 19 cols
  investor_transactions           32,778 rows x 13 cols
  portfolio_holdings                 322 rows x 8 cols
  benchmark_indices                8,050 rows x 3 cols
Loaded 10/10 datasets


In [10]:
pre = [{"dataset":n,"rows":len(d),"nulls":int(d.isnull().sum().sum()),"dups":int(d.duplicated().sum())} for n,d in raw.items()]
display(pd.DataFrame(pre))

,dataset,rows,nulls,dups
0,fund_master,40,0,0
1,nav_history,46000,0,0
2,aum_by_fund_house,90,0,0
3,monthly_sip_inflows,48,12,0
4,category_inflows,144,0,0
5,industry_folio_count,21,0,0
6,scheme_performance,40,0,0
7,investor_transactions,32778,0,0
8,portfolio_holdings,322,0,0
9,benchmark_indices,8050,0,0


In [11]:
clean = {}

if "nav_history" in raw:
    df = raw["nav_history"].copy()
    before = len(df)
    df = df.dropna(subset=["amfi_code","date","nav"])
    df["nav"] = pd.to_numeric(df["nav"], errors="coerce")
    df = df[df["nav"] > 0].drop_duplicates(subset=["amfi_code","date"])
    df = df.sort_values(["amfi_code","date"]).reset_index(drop=True)

    codes    = df["amfi_code"].unique()
    full_idx = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    midx     = pd.MultiIndex.from_product([codes, full_idx], names=["amfi_code","date"])

    df = (df.set_index(["amfi_code","date"]).reindex(midx)["nav"]
          .groupby(level=0).ffill().reset_index().dropna(subset=["nav"]))

    clean["nav_history"] = df
    print(f"nav_history: {before:,} raw -> {len(df):,} after ffill ({len(df)-before:,} holiday rows added)")
    display(df.head(3))

nav_history: 46,000 raw -> 64,320 after ffill (18,320 holiday rows added)


,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239


In [12]:
if "fund_master" in raw:
    df = raw["fund_master"].copy()
    df = df.drop_duplicates(subset=["amfi_code"])
    df["amfi_code"]         = df["amfi_code"].astype(str).str.strip()
    df["fund_house"]        = df["fund_house"].str.strip().str.title()
    df["scheme_name"]       = df["scheme_name"].str.strip()
    df["category"]          = df["category"].str.strip().str.title()
    df["sub_category"]      = df["sub_category"].str.strip().str.title()
    df["plan"]              = df["plan"].str.strip().str.title()
    df["risk_category"]     = df["risk_category"].str.strip().str.title()
    df["expense_ratio_pct"] = pd.to_numeric(df["expense_ratio_pct"], errors="coerce")
    df["exit_load_pct"]     = pd.to_numeric(df["exit_load_pct"], errors="coerce").fillna(0.0)

    clean["fund_master"] = df

    print(f"fund_master: {len(raw['fund_master'])} -> {len(df)} rows")
    display(df["risk_category"].value_counts().to_frame("count"))

fund_master: 40 -> 40 rows


,count
risk_category,
Moderate,16
High,8
Very High,6
Low,6
Moderately High,4


In [13]:
if "investor_transactions" in raw:
    df = raw["investor_transactions"].copy()
    before = len(df)
    df = df.dropna(subset=["investor_id","amfi_code","transaction_date"])
    df["amount_inr"] = pd.to_numeric(df["amount_inr"], errors="coerce")
    df = df[df["amount_inr"] > 0]
    df["transaction_type"] = df["transaction_type"].str.strip().str.title()
    df["transaction_type"] = df["transaction_type"].replace({"Sip":"SIP","Lump Sum":"Lumpsum"})
    df["kyc_status"] = df["kyc_status"].str.strip().str.title()
    df["city_tier"]  = df["city_tier"].str.strip().str.upper()
    df = df.sort_values(["investor_id","transaction_date"]).reset_index(drop=True)
    clean["investor_transactions"] = df
    print(f"investor_transactions: {before:,} -> {len(df):,} rows")
    display(df["transaction_type"].value_counts().to_frame("count"))

investor_transactions: 32,778 -> 32,778 rows


,count
transaction_type,
SIP,19716
Lumpsum,8095
Redemption,4967


In [14]:
if "scheme_performance" in raw:
    df = raw["scheme_performance"].copy()
    df = df.drop_duplicates(subset=["amfi_code"])

    for col in [
        "return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
        "benchmark_3yr_pct", "alpha", "beta", "sharpe_ratio",
        "sortino_ratio", "std_dev_ann_pct", "max_drawdown_pct",
        "aum_crore", "expense_ratio_pct"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    clean["scheme_performance"] = df

    print(f"scheme_performance: {len(df)} rows")
    print(f"  Negative Sharpe funds: {(df['sharpe_ratio'] < 0).sum()} (valid — retained)")

    display(
        df[["sharpe_ratio", "alpha", "beta", "max_drawdown_pct"]]
        .describe()
        .round(4)
    )

scheme_performance: 40 rows
  Negative Sharpe funds: 0 (valid — retained)


,sharpe_ratio,alpha,beta,max_drawdown_pct
count,40.0000,40.0000,40.0000,40.0000
mean,1.3618,1.2535,0.8733,-19.2002
std,1.4758,0.4474,0.2248,8.8192
min,0.8000,0.5100,0.2200,-33.5000
25%,0.8650,0.8875,0.8900,-25.0625
50%,0.9250,1.2050,0.9600,-20.6000
75%,0.9850,1.7000,1.0000,-14.2550
max,7.6800,1.9800,1.0400,-2.2300


In [15]:
for name in ["aum_by_fund_house","monthly_sip_inflows","category_inflows",
              "industry_folio_count","portfolio_holdings","benchmark_indices"]:
    if name in raw:
        clean[name] = raw[name].copy()

print(f"Total datasets cleaned: {len(clean)}")

Total datasets cleaned: 10


In [16]:
rows = []
for name in DATASETS:
    if name not in raw or name not in clean:
        continue
    b, a = raw[name], clean[name]
    rows.append({"dataset":name,"rows_before":len(b),"rows_after":len(a),
                 "removed":len(b)-len(a),"nulls_before":int(b.isnull().sum().sum()),
                 "nulls_after":int(a.isnull().sum().sum())})

report = pd.DataFrame(rows)
display(report)
report.to_csv(PROCESSED_DIR / "cleaning_report.csv", index=False)
print("cleaning_report.csv saved")

,dataset,rows_before,rows_after,removed,nulls_before,nulls_after
0,fund_master,40,40,0,0,0
1,nav_history,46000,64320,-18320,0,0
2,aum_by_fund_house,90,90,0,0,0
3,monthly_sip_inflows,48,48,0,12,12
4,category_inflows,144,144,0,0,0
5,industry_folio_count,21,21,0,0,0
6,scheme_performance,40,40,0,0,0
7,investor_transactions,32778,32778,0,0,0
8,portfolio_holdings,322,322,0,0,0
9,benchmark_indices,8050,8050,0,0,0


cleaning_report.csv saved


In [17]:
for name, df in clean.items():
    path = PROCESSED_DIR / f"clean_{name}.csv"
    df.to_csv(path, index=False)
    print(f"  clean_{name}.csv  ({len(df):,} rows)")

  clean_nav_history.csv  (64,320 rows)
  clean_fund_master.csv  (40 rows)
  clean_investor_transactions.csv  (32,778 rows)
  clean_scheme_performance.csv  (40 rows)
  clean_aum_by_fund_house.csv  (90 rows)
  clean_monthly_sip_inflows.csv  (48 rows)
  clean_category_inflows.csv  (144 rows)
  clean_industry_folio_count.csv  (21 rows)
  clean_portfolio_holdings.csv  (322 rows)
  clean_benchmark_indices.csv  (8,050 rows)


In [18]:
engine = create_engine(f"sqlite:///{DB_PATH}")

with engine.begin() as conn:
    for name in ["fund_master","nav_history","aum_by_fund_house",
                 "monthly_sip_inflows","category_inflows","industry_folio_count",
                 "scheme_performance","investor_transactions",
                 "portfolio_holdings","benchmark_indices"]:
        if name not in clean:
            continue
        df = clean[name].copy()
        for col in df.select_dtypes(include=["datetime64[ns]"]).columns:
            df[col] = df[col].astype(str)
        df.to_sql(name, conn, if_exists="replace", index=False)
        print(f"  {name:<35} {len(df):>7,} rows loaded")

    for stmt in [
        "CREATE INDEX IF NOT EXISTS idx_nav_code_date   ON nav_history(amfi_code, date)",
        "CREATE INDEX IF NOT EXISTS idx_txn_investor    ON investor_transactions(investor_id)",
        "CREATE INDEX IF NOT EXISTS idx_txn_code_date   ON investor_transactions(amfi_code, transaction_date)",
        "CREATE INDEX IF NOT EXISTS idx_bench_name_date ON benchmark_indices(index_name, date)",
    ]:
        conn.execute(text(stmt))

print(f"Database ready: {DB_PATH}")

  fund_master                              40 rows loaded
  nav_history                          64,320 rows loaded
  aum_by_fund_house                        90 rows loaded
  monthly_sip_inflows                      48 rows loaded
  category_inflows                        144 rows loaded
  industry_folio_count                     21 rows loaded
  scheme_performance                       40 rows loaded
  investor_transactions                32,778 rows loaded
  portfolio_holdings                      322 rows loaded
  benchmark_indices                     8,050 rows loaded
Database ready: D:\Engsci_Study_materials\INTERNSHIP\Capstone_Project\data\db\bluestock_mf.db


In [19]:
QUERIES = {
    "Q1 - Top 5 funds by AUM": "SELECT f.scheme_name, f.fund_house, ROUND(p.aum_crore,2) AS aum_crore, ROUND(p.return_3yr_pct,2) AS ret_3yr, ROUND(p.sharpe_ratio,4) AS sharpe FROM scheme_performance p JOIN fund_master f ON f.amfi_code=p.amfi_code ORDER BY p.aum_crore DESC LIMIT 5",
    "Q2 - Avg NAV per month": "SELECT SUBSTR(date,1,7) AS month, ROUND(AVG(nav),2) AS avg_nav, COUNT(DISTINCT amfi_code) AS funds FROM nav_history GROUP BY month ORDER BY month LIMIT 12",
    "Q3 - SIP inflow YoY": "SELECT SUBSTR(month,1,4) AS year, ROUND(SUM(sip_inflow_crore),2) AS total_sip_cr, ROUND(AVG(yoy_growth_pct),2) AS avg_yoy FROM monthly_sip_inflows GROUP BY year ORDER BY year",
    "Q4 - Transactions by state": "SELECT state, city_tier, COUNT(*) AS txns, ROUND(SUM(amount_inr),0) AS total_inr FROM investor_transactions GROUP BY state,city_tier ORDER BY total_inr DESC LIMIT 10",
    "Q5 - Expense ratio < 1%": "SELECT scheme_name, fund_house, category, ROUND(expense_ratio_pct,4) AS ter FROM fund_master WHERE expense_ratio_pct < 1.0 ORDER BY expense_ratio_pct",
    "Q6 - Sharpe ratio ranking": "SELECT f.scheme_name, f.risk_category, ROUND(p.sharpe_ratio,4) AS sharpe, ROUND(p.alpha,4) AS alpha, ROUND(p.max_drawdown_pct,2) AS max_dd FROM scheme_performance p JOIN fund_master f ON f.amfi_code=p.amfi_code ORDER BY p.sharpe_ratio DESC LIMIT 10",
    "Q7 - Monthly txn volume": "SELECT SUBSTR(transaction_date,1,7) AS month, transaction_type, COUNT(*) AS count, ROUND(SUM(amount_inr),0) AS total_inr FROM investor_transactions GROUP BY month,transaction_type ORDER BY month LIMIT 15",
    "Q8 - Top portfolio holdings": "SELECT stock_name, sector, COUNT(DISTINCT amfi_code) AS funds, ROUND(AVG(weight_pct),2) AS avg_wt FROM portfolio_holdings GROUP BY stock_name,sector ORDER BY funds DESC,avg_wt DESC LIMIT 10",
    "Q9 - Benchmark total return": "SELECT index_name, ROUND(MIN(close_value),2) AS min_val, ROUND(MAX(close_value),2) AS max_val, ROUND((MAX(close_value)-MIN(close_value))/MIN(close_value)*100,2) AS total_ret_pct FROM benchmark_indices GROUP BY index_name ORDER BY total_ret_pct DESC",
    "Q10 - Age group vs SIP amount": "SELECT age_group, gender, COUNT(DISTINCT investor_id) AS investors, ROUND(AVG(amount_inr),0) AS avg_sip_inr FROM investor_transactions WHERE transaction_type='SIP' GROUP BY age_group,gender ORDER BY age_group,gender",
}

with engine.connect() as conn:
    for label, q in QUERIES.items():
        print(f"{'─'*55}{label}{'─'*55}")
        display(pd.read_sql_query(text(q), conn))

───────────────────────────────────────────────────────Q1 - Top 5 funds by AUM───────────────────────────────────────────────────────


,scheme_name,fund_house,aum_crore,ret_3yr,sharpe
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset Mf,"49,046.0000",14.5600,0.9100
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra Mf,"47,469.0000",18.2300,0.9600
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India Mf,"43,630.0000",20.1500,0.8100
3,DSP Top 100 Equity Fund - Regular - Growth,Dsp Mutual Fund,"41,828.0000",12.8200,0.9200
4,UTI Mid Cap Fund - Regular - Growth,Uti Mutual Fund,"41,728.0000",15.6100,0.8200


───────────────────────────────────────────────────────Q2 - Avg NAV per month───────────────────────────────────────────────────────


,month,avg_nav,funds
0,2022-01,207.0500,40
1,2022-02,207.7200,40
2,2022-03,209.6900,40
3,2022-04,211.7900,40
4,2022-05,212.7300,40
5,2022-06,213.8600,40
6,2022-07,213.9500,40
7,2022-08,215.7000,40
8,2022-09,218.4700,40
9,2022-10,219.5300,40


───────────────────────────────────────────────────────Q3 - SIP inflow YoY───────────────────────────────────────────────────────


...

,stock_name,sector,funds,avg_wt
0,Bharti Airtel Ltd,Telecom,15,9.7100
1,Grasim Industries Ltd,Diversified,14,12.0900
2,NTPC Ltd,Utilities,13,11.9300
3,HCL Technologies Ltd,IT,13,11.3200
4,ICICI Bank Ltd,Banking,13,9.3900
5,Reliance Industries Ltd,Energy,13,9.0700
6,Axis Bank Ltd,Banking,12,12.5600
7,Tata Motors Ltd,Automobile,12,9.5600
8,UltraTech Cement Ltd,Cement,12,8.7500
9,Adani Ports & SEZ Ltd,Infrastructure,12,8.4700


───────────────────────────────────────────────────────Q9 - Benchmark total return───────────────────────────────────────────────────────


,index_name,min_val,max_val,total_ret_pct
0,NIFTY_MIDCAP150,"8,980.6000","32,990.6600",267.3500
1,BSE_SMALLCAP,"23,592.6400","79,075.3900",235.1700
2,NIFTY500,"14,426.0400","38,418.8700",166.3200
3,CRISIL_GILT,"1,444.1300","2,302.7900",59.4600
4,NIFTY50,"17,492.7900","27,798.7200",58.9200
5,NIFTY100,"14,128.8600","21,088.5800",49.2600
6,CRISIL_LIQUID,"2,281.5100","3,046.0000",33.5100


───────────────────────────────────────────────────────Q10 - Age group vs SIP amount───────────────────────────────────────────────────────


,age_group,gender,investors,avg_sip_inr
0,18-25,Female,237,"10,969.0000"
1,18-25,Male,469,"10,944.0000"
2,26-35,Female,632,"10,948.0000"
3,26-35,Male,1305,"11,006.0000"
4,36-45,Female,393,"10,778.0000"
5,36-45,Male,792,"10,938.0000"
6,46-55,Female,195,"10,898.0000"
7,46-55,Male,369,"11,254.0000"
8,56+,Female,132,"10,691.0000"
9,56+,Male,238,"12,069.0000"


In [20]:
with engine.connect() as conn:
    tables = pd.read_sql_query(
        text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"),
        conn
    )

    print("Tables and row counts:")
    for t in tables["name"]:
        n = pd.read_sql_query(
            text(f"SELECT COUNT(*) AS n FROM {t}"),
            conn
        )["n"].iloc[0]

        print(f"  {t:<35} {n:>8,} rows")

    print("Key data quality checks:")
    checks = {
        "NAV rows with nav <= 0":
            "SELECT COUNT(*) AS n FROM nav_history WHERE CAST(nav AS REAL) <= 0",

        "Transactions amount <= 0":
            "SELECT COUNT(*) AS n FROM investor_transactions WHERE amount_inr <= 0",

        "Null amfi_codes in nav":
            "SELECT COUNT(*) AS n FROM nav_history WHERE amfi_code IS NULL",
    }

    for label, q in checks.items():
        n = pd.read_sql_query(text(q), conn)["n"].iloc[0]
        flag = "PASS" if n == 0 else "WARN"
        print(f"  [{flag}] {label:<42} {n:>6,}")

Tables and row counts:
  aum_by_fund_house                         90 rows
  benchmark_indices                      8,050 rows
  category_inflows                         144 rows
  dim_date                                   0 rows
  dim_fund                                   0 rows
  fact_aum                                   0 rows
  fact_benchmark                             0 rows
  fact_category_inflows                      0 rows
  fact_folio_count                           0 rows
  fact_nav                                   0 rows
  fact_performance                           0 rows
  fact_portfolio                             0 rows
  fact_sip_industry                          0 rows
  fact_transactions                          0 rows
  fund_master                               40 rows
  industry_folio_count                      21 rows
  investor_transactions                 32,778 rows
  monthly_sip_inflows                       48 rows
  nav_history                           6